#Wczytywanie plików

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!unzip dataset.zip

In [ ]:
!unzip train.zip -d dataset

In [ ]:
!unzip test.zip -d dataset

#Import bibliotek

In [ ]:
# Biblioteki numeryczne i wizualizacyjne
import torch
import random
import numpy as np
import matplotlib.pyplot as plt

# Przetwarzanie obrazów
from PIL import Image

# Narzędzia PyTorch do pracy z danymi
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torchvision.transforms.functional as F

# Moduły do budowy sieci oraz walidacji
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import KFold
from torch.utils.data import SubsetRandomSampler


#Parametry eksperymentu

In [ ]:
# Ścieżka do zbioru danych
DATASET_PATH = "dataset"

# Docelowy rozmiar obrazów
IMAGE_SIZE = (128, 128)

# Rozmiar batcha
BATCH_SIZE = 32

#Obliczanie statystyk normalizacji (mean i std)

In [ ]:
# Transformacja bazowa wykorzystywana wyłącznie do obliczenia statystyk
basic_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor()
])

# Tymczasowy zbiór danych bez augmentacji
train_temp = datasets.ImageFolder(
    root=f"{DATASET_PATH}/train",
    transform=basic_transform
)

loader = DataLoader(train_temp, batch_size=32, shuffle=False)

# Inicjalizacja tensorów na średnią i odchylenie standardowe
mean = torch.zeros(3)
std = torch.zeros(3)
n_samples = 0

# Iteracja po całym zbiorze treningowym
for images, _ in loader:
    batch_size = images.size(0)
    images = images.view(batch_size, images.size(1), -1)

    mean += images.mean(dim=2).sum(dim=0)
    std += images.std(dim=2).sum(dim=0)
    n_samples += batch_size

# Uśrednienie statystyk
mean /= n_samples
std /= n_samples

print("Mean:", mean)
print("Std:", std)


#Pipeline danych - zbiór treningowy (augmentacja)

In [ ]:
train_transform = transforms.Compose([
    # Skalowanie obrazów do wspólnego rozmiaru
    transforms.Resize(IMAGE_SIZE),

    # Losowa rotacja w pełnym zakresie
    transforms.RandomRotation(degrees=(0, 360)),

    # Flipy
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),

    # Przesunięcie i skalowanie
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.8, 1.2)
    ),

    # Zmiany oświetlenia i koloru
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.3,
        hue=0.1
    ),

    # Konwersja obrazu do tensora
    transforms.ToTensor(),

    # Dodanie niewielkiego szumu gaussowskiego
    transforms.Lambda(
        lambda x: torch.clamp(
            x + torch.randn_like(x) * 0.03,
            0.0,
            1.0
        )
    ),

    # Normalizacja danych
    transforms.Normalize(mean=mean, std=std)
])

#Pipeline danych - zbiór testowy

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

#Wczytanie danych

In [ ]:

train_dataset = datasets.ImageFolder(
    root=f"{DATASET_PATH}/train",
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    root=f"{DATASET_PATH}/test",
    transform=test_transform
)
###############################Dataset bez augumentacji (do walidacji w KFold)

train_dataset_plain = datasets.ImageFolder(
    root=f"{DATASET_PATH}/train",
    transform=test_transform
)
################################
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

#Manualny pipeline przetwarzania - implementacja poglądowa

In [ ]:
def manual_pipeline(img_path, mean, std, train=True):
    """
    Funkcja realizująca ręcznie proces augmentacji i normalizacji obrazu.
    Stanowi alternatywę dla transforms.Compose
    """
    img = Image.open(img_path).convert("RGB")

    # Skalowanie obrazów do wspólnego rozmiaru
    img = F.resize(img, IMAGE_SIZE)

    if train:

        # Losowa rotacja w pełnym zakresie
        img = F.rotate(img, random.uniform(0, 360))

        # Flipy
        if random.random() > 0.5:
            img = F.hflip(img)
        if random.random() > 0.5:
            img = F.vflip(img)

        # Przesunięcie i skalowanie
        scale = random.uniform(0.8, 1.2)
        w, h = img.size
        img = F.resize(img, (int(h * scale), int(w * scale)))
        img = F.center_crop(img, IMAGE_SIZE)

        # Zmiany oświetlenia i koloru
        img = F.adjust_brightness(img, random.uniform(0.7, 1.3))
        img = F.adjust_contrast(img, random.uniform(0.7, 1.3))
        img = F.adjust_saturation(img, random.uniform(0.7, 1.3))
        img = F.adjust_hue(img, random.uniform(-0.1, 0.1))

    # Konwersja obrazu do tensora
    img = F.to_tensor(img)

    if train:

        # Dodanie niewielkiego szumu gaussowskiego
        img = torch.clamp(img + torch.randn_like(img) * 0.03, 0, 1)

    # Normalizacja danych
    img = F.normalize(img, mean.tolist(), std.tolist())

    return img

Wizualizacja danych

In [ ]:
# Funkcja odwracająca normalizację w celu poprawnej wizualizacji obrazu
def denormalize(img, mean, std):
    return img * std.view(3,1,1) + mean.view(3,1,1)

In [ ]:
def visualize(loader):
    images, labels = next(iter(loader))
    images = denormalize(images, mean, std)

    fig, axs = plt.subplots(1, 5, figsize=(15, 3))
    for i in range(5):
        img = images[i].permute(1, 2, 0).numpy()
        axs[i].imshow(img)
        axs[i].axis("off")

    plt.show()

visualize(train_loader)

# Budowa konwolucyjnej sieci neuronowej

In [ ]:
# Definicja modelu
class CNN(nn.Module):
    def __init__(self, num_classes = 3):
        super(CNN, self).__init__()

        # Ekstrakcja cech (szukanie krawędzi, kształtów)
        self.features = nn.Sequential(
            # Blok 1: wejście RGB (3 kanały) -> uczenie się 32 filtrów
            nn.Conv2d(in_channels = 3, out_channels = 32, kernel_size = 3,
                      padding = 1),

            # Funkcja aktywacji ReLU
            nn.ReLU(),

            # Redukcja: 128x128 -> 64x64
            nn.MaxPool2d(kernel_size = 2, stride = 2),

            # Blok 2: 32 -> 64 filtry
            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.ReLU(),

            # Redukcja: 64x64 -> 32x32
            nn.MaxPool2d(2, 2),

            # Blok 3: 64 -> 128 filtrów
            nn.Conv2d(64, 128, kernel_size = 3, padding = 1),
            nn.ReLU(),

            # Redukcja: 32x32 -> 16x16
            nn.MaxPool2d(2, 2)
        )

        # Klasyfikator - podjęcie decyzji
        self.classifier = nn.Sequential(
            nn.Flatten(), # Spłaszczenie: 128 kanałów * 16 * 16 pixeli
            nn.Linear(128 * 16 * 16, 512),
            nn.ReLU(),       # Funkcja aktywacji
            nn.Dropout(0.5), # Wyłączenie losowo 50% (przeciw przeuczeniu)
            nn.Linear(512, num_classes) # Wyjście: 3 klasy (1gr, 5gr, 1zł)
        )

    # Funkcja, która wykonuje się po odebraniu danych przez model
    def forward(self, x):
        x = self.features(x)    # Wydobądź cechy z obrazka
        x = self.classifier(x)  # Podejmij decyzję
        return x

# Konfiguracja treningu, walidacja 5-krotna

In [ ]:
K_FOLDS = 5           # Podział na 5 części
NUM_EPOCHS = 60       # Liczba epok na każdy fold
LEARNING_RATE = 0.001 # Krok uczenia

num_classes = len(train_dataset.classes)  # 3 monety

# Funkcja straty
criterion = nn.CrossEntropyLoss()

# Wywołanie walidacji 5-krotnej
kfold = KFold(n_splits = K_FOLDS, shuffle = True, random_state = 42)
results = {}

print(f"\nRozpoczęcie walidacji krzyżowej:")

# Pętle walidacji krzyżowej

for fold, (train_ids, val_ids) in enumerate(kfold.split(train_dataset)):
    print(f"\nFOLD {fold + 1} / {K_FOLDS} ")

    # Wybór konkretnych zdjęć dla danego folda
    train_subsampler = SubsetRandomSampler(train_ids)
    val_subsampler = SubsetRandomSampler(val_ids)

    trainloader_fold = DataLoader(train_dataset, batch_size = BATCH_SIZE,
                                  sampler = train_subsampler)
    #################################################################
    valloader_fold = DataLoader(train_dataset_plain, batch_size=BATCH_SIZE,
                                sampler=val_subsampler)

    #valloader_fold = DataLoader(train_dataset, batch_size = BATCH_SIZE,
                                #sampler = val_subsampler)
#############################################################################
    # Inicjalizacja nowego modelu dla każdego folda (reset wag)
    model = CNN(num_classes = num_classes)

    # Algorytm optymalizacji
    optimizer = optim.Adam(model.parameters(), lr = LEARNING_RATE)

    # Pętla epok
    for epoch in range(NUM_EPOCHS):
        # Faza treningu

        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for inputs, labels in trainloader_fold:

            optimizer.zero_grad()       # Reset starych obliczeń (gradientów)
            outputs = model(inputs)     # Predykcja modelu
            loss = criterion(outputs, labels) # Obliczenie błędu
            loss.backward()             # Wsteczna propagacja
            optimizer.step()            # Aktualizacja wag

            # Statystyki do wyświetlenia
            running_loss += loss.item()

            # Wybór klasy z największym prawdopodobieństwem
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)

            # Ilość dobrze odgadniętych klas
            correct_train += (predicted == labels).sum().item()

        train_acc = 100 * correct_train / total_train

        # Faza walidacji

        model.eval()
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for inputs, labels in valloader_fold:
                outputs = model(inputs)
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        # Wypisanie wyniku co epokę
        val_acc = 100 * correct_val / total_val
        print(f"- Epoka {epoch+1}: Strata: {running_loss / len(trainloader_fold):.4f}")
        print(f"Train Acc: {train_acc:.1f}% | Val Acc: {val_acc:.1f}%")

    results[fold] = val_acc
    print(f"\nWynik końcowy dla Fold {fold + 1}: {val_acc:.2f}%")

# Podsumowanie średniej skuteczności modelu
print(f"\nŚrednia dokładność: {sum(results.values())/len(results):.2f}%")

# Finalny trening modelu

In [ ]:
print("\nFinalny trening modelu na całym zbiorze treningowym:\n")
# Trenowanie nowego modelu na wszystkich danych treningowych

final_model = CNN(num_classes = num_classes)
optimizer = optim.Adam(final_model.parameters(), lr = LEARNING_RATE)
#####################################################################
###################################################################
# zapisywanie historii dokładności (dla wykresów)
train_acc_history = []
#val_acc_history = []

###################################################################
#################################################################
for epoch in range(NUM_EPOCHS):
  ######################################################################
  #######################################################################
    #final_model.train()
    #running_loss = 0.0

    final_model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0


    ###############################################################
    ################################################################
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = final_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        ####################################################################
        ###################################################################
        #Do wykresu - liczenie acc w trakcie treningu
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    train_acc = 100 * correct_train / total_train

    train_acc_history.append(train_acc)
        ####################################################################
        ####################################################################
    print(f"Epoka {epoch + 1} zakończona. Strata: {running_loss / len(train_loader):.4f}")



# Ewalucja modelu na zbiorze testowym
final_model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = final_model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_acc = 100 * correct / total
print(f"\nDokładność modelu na zbiorze testowym: {test_acc:.2f}%")

# Wizualizacja predykcji modelu

In [ ]:
def visualize(model, loader):
    model.eval()

    # Pobranie paczki zdjęć
    images, labels = next(iter(loader))

    outputs = model(images)
    _, preds = torch.max(outputs, 1)

    # Odwrócenie normalizacji
    images = denormalize(images, mean, std)

    # Ustawienia wykresu
    fig = plt.figure(figsize = (16, 8))
    rows = 4
    cols = 8

    # Rysowanie obrazków
    for i in range(min(len(images), rows * cols)):
        ax = fig.add_subplot(rows, cols, i + 1, xticks = [], yticks = [])
        img = images[i].permute(1, 2, 0).numpy() # Zamiana wymiarów dla matplotlib

        ax.imshow(img)

        # Logika kolorów
        title_color = "green" if preds[i] == labels[i] else "red"

        pred_label = test_dataset.classes[preds[i]]
        true_label = test_dataset.classes[labels[i]]

        ax.set_title(f"P: {pred_label}\nT: {true_label}",
                     color = title_color, fontsize = 10)

    plt.tight_layout()
    plt.show()

visualize(final_model, test_loader)

#Raport klasyfikacji


In [ ]:
#Raport klasyfikacji
from sklearn.metrics import classification_report
import torch


final_model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = final_model(images)
        _, predicted = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

class_names = test_dataset.classes


print("RAPORT KLASYFIKACJI:\n")


print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4
    )
)

#Confusion matrix (macierz pomyłek)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

plt.figure(figsize=(6, 6))
disp.plot(values_format="d")
plt.title("Confusion Matrix (Test)")
plt.xticks(rotation=45)
plt.show()

# wersja procentowa (normalizacja po wierszach)
cm_norm = confusion_matrix(y_true, y_pred, normalize="true")

disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=class_names)

plt.figure(figsize=(6, 6))
disp_norm.plot(values_format=".2f")
plt.title("Confusion Matrix (Test) – normalized")
plt.xticks(rotation=45)
plt.show()

#Wykres dokładności

In [ ]:

epochs = range(1, len(train_acc_history) + 1)

plt.figure(figsize=(10, 6))

# Train accuracy
plt.plot(
    epochs,
    train_acc_history,
    linewidth=2,
    label="Train Accuracy"
)

# Test accuracy – linia pozioma
plt.hlines(
    y=test_acc,
    xmin=1,
    xmax=len(train_acc_history),
    colors="red",
    linestyles="dashed",
    label=f"Test Accuracy = {test_acc:.2f}%"
)

plt.xlabel("Epoka")
plt.ylabel("Dokładność [%]")
plt.title("Dokładność modelu: trening vs test")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
